In [ ]:
# =============================================================================
# Nine18 ARC-AGI-3 actual scored run — exact bundled agents
#
# This notebook intentionally stays close to the original TAAF/Kaggle harness:
#   * load deploy_target.pkl exactly from the attached TAAF bundle,
#   * load benchmark_initial.pkl exactly from the attached TAAF bundle,
#   * do not replace, wrap, monkey-patch, or synthesize bm.solver / agents,
#   * use the live Kaggle gateway only when KAGGLE_IS_COMPETITION_RERUN is true,
#   * let the bundle's official teardown create the real submission.parquet.
#
# Non-rerun execution is treated only as offline validation. It is not a scored
# result and should not be interpreted as leaderboard performance.
# =============================================================================

import inspect
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Marker only: no code below consumes this to inject or replace agents.
os.environ["NINE18_EXACT_BUNDLED_AGENTS"] = "1"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")


# --- FIX 1 (new) -------------------------------------------------------------
# Surface a GPU/wheelhouse mismatch loudly instead of letting vLLM fail silently
# later. The bundled wheelhouse (arc3-vllm-h100-wheelhouse-v3) is built for H100;
# ARC-AGI-3's free accelerator pool is RTX 6000 (g4-standard-48). If the wrong
# GPU is attached, this print is the only thing that will tell you why setup
# failed, since minimal diagnostics suppresses almost everything else.
try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).strip().split("\n")[0]
except Exception as exc:
    gpu_name = f"<nvidia-smi failed: {exc}>"
print(f"taaf.kaggle: attached GPU = {gpu_name}")
if "H100" not in gpu_name and TRUE_SUBMISSION:
    print(
        f"taaf.kaggle: WARNING - wheelhouse is H100-built but attached GPU is '{gpu_name}'. "
        "If vLLM fails to load kernels below, this is why."
    )
# --- end FIX 1 -----------------------------------------------------------------


# =============================================================================
# 2. Install the ARC runtime
#
# Install `arc-agi` from the offline competition wheelhouse (the Kaggle
# submission environment has no internet).
# =============================================================================

# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)


# =============================================================================
# 3. Locate the source bundle
#
# Find the uploaded TAAF source dataset by its marker file, and record where
# Kaggle mounted every attached input so setup commands and the solver can
# find them.
# =============================================================================

# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


# =============================================================================
# 4. Import the bundled source and run solver setup
#
# Put the snapshotted repositories on the path (this process and any child
# processes), then run the solver's setup commands - installing wheels,
# fetching model weights, and so on.
# =============================================================================

# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# --- FIX 2 (changed) -----------------------------------------------------------
# Original used `subprocess.run(..., check=True)`: a single transient failure
# (e.g. vLLM server failing to bind on first try) killed the whole notebook
# before the benchmark ever loaded, producing a hard-zero submission. Retry
# each setup command up to 3 times with backoff; only raise after all 3 fail.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    result = None
    for attempt in range(1, 4):
        print(f"taaf.kaggle: setup command (attempt {attempt}/3): {command}", flush=True)
        result = subprocess.run(command, shell=True, cwd=WORKING_DIR, env=env)
        if result.returncode == 0:
            break
        print(f"taaf.kaggle: setup command failed (exit {result.returncode}), retrying...", flush=True)
        time.sleep(10 * attempt)
    else:
        raise subprocess.CalledProcessError(result.returncode, command)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)
# --- end FIX 2 -------------------------------------------------------------

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)


# =============================================================================
# 5. Load the benchmark
#
# Unpickle the deployment target and the benchmark, stamping the real
# submission state onto the target and pointing the benchmark's outputs at
# the Kaggle working directory.
# =============================================================================

# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


# =============================================================================
# 6. Exact-agent integrity manifest
#
# No customization hook is used here. The benchmark's bundled solver / agent
# object remains exactly the unpickled object from benchmark_initial.pkl.
# This section only records what was loaded so a failed run can be audited.
# =============================================================================


def _safe_obj_file(obj):
    try:
        return inspect.getfile(obj.__class__)
    except Exception as exc:
        return f"<unavailable: {exc}>"


def _safe_obj_module(obj):
    try:
        return obj.__class__.__module__
    except Exception as exc:
        return f"<unavailable: {exc}>"


def _safe_obj_class(obj):
    try:
        return obj.__class__.__qualname__
    except Exception as exc:
        return f"<unavailable: {exc}>"


exact_agent_manifest = {
    "true_submission": TRUE_SUBMISSION,
    "bundle_dir": str(BUNDLE_DIR),
    "benchmark_class": _safe_obj_class(bm),
    "benchmark_module": _safe_obj_module(bm),
    "benchmark_file": _safe_obj_file(bm),
    "solver_class": _safe_obj_class(getattr(bm, "solver", None)),
    "solver_module": _safe_obj_module(getattr(bm, "solver", None)),
    "solver_file": _safe_obj_file(getattr(bm, "solver", None)),
    "target_class": _safe_obj_class(target),
    "target_module": _safe_obj_module(target),
    "target_file": _safe_obj_file(target),
    "solver_repr": repr(getattr(bm, "solver", None))[:1000],
}
(WORKING_DIR / "exact_agent_manifest.json").write_text(
    json.dumps(exact_agent_manifest, indent=2, sort_keys=True) + "\n"
)
print("taaf.kaggle: exact bundled solver =", exact_agent_manifest["solver_class"], exact_agent_manifest["solver_module"])
print("taaf.kaggle: wrote exact_agent_manifest.json")


# =============================================================================
# 7. Run the benchmark
#
# In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the
# Kaggle gateway and play the live competition Arcade. Otherwise - an
# interactive "Save & Run" - play the competition's bundled environment files
# offline, with no gateway required, so the notebook runs end-to-end without
# a submission. Teardown commands run afterward even if the run raises.
# =============================================================================

# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

# NOTE ON n_passes: raising this to retry stochastic rollouts only helps if
# bm.run()'s scoring takes the BEST score per game across passes rather than
# averaging them - and it costs Nx runtime inside the 9h Kaggle ceiling. Left
# at 1 here because that scoring behaviour lives inside taaf.benchmark, which
# isn't visible in this harness; confirm it there before changing this value.
bm.n_passes = 1
bm.game_weights = None

# --- FIX 3 (changed) -------------------------------------------------------
# Original gated the soft deadline entirely behind `if not TRUE_SUBMISSION`,
# so a real competition rerun had NO graceful exit at all - a stuck game
# could burn the full 9h Kaggle ceiling, the process gets killed, and neither
# teardown_commands nor a submission.parquet ever run. Both branches now get
# a soft_end: interactive runs keep the original margin, and real submissions
# get a margin sized to leave enough time for teardown + parquet write.
KAGGLE_HARD_LIMIT_S = 9 * 3600
budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
if TRUE_SUBMISSION:
    budget = budget or KAGGLE_HARD_LIMIT_S
    margin = min(900.0, budget * 0.1)  # bail early so teardown + parquet write actually finish
else:
    margin = min(600.0, budget / 2) if budget > 0 else 0.0
soft_end = (
    datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - margin)
    if budget > 0 else None
)
print(f"taaf.kaggle: soft_end = {soft_end} (budget={budget}s, margin={margin}s)")
# --- end FIX 3 ---------------------------------------------------------------

# Play the benchmark; teardown commands run even if the run raises.
try:
    result = await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)

    # --- FIX 4 (new) ---------------------------------------------------------
    # Under TAAF_MINIMAL_DIAGNOSTICS, virtually nothing gets written for a real
    # submission, so a bad score is unexplainable after the fact - no way to
    # tell setup failure from solver failure from a hard game rotation. Write
    # a tiny best-effort summary regardless of mode. `result`'s actual shape
    # is not visible in this harness (lives in taaf.benchmark); repr() is a
    # deliberately conservative fallback that can't itself raise on unknown
    # attributes. Confirm the real return type there and tighten this once known.
    try:
        summary = {
            "true_submission": TRUE_SUBMISSION,
            "result_repr": repr(result)[:2000],
            "games": [getattr(g, "env_name", repr(g)) for g in getattr(bm, "games", [])],
            "n_passes": bm.n_passes,
        }
        (WORKING_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))
        print("taaf.kaggle: wrote run_summary.json")
    except Exception as diag_exc:
        print(f"taaf.kaggle: summary dump skipped: {diag_exc}")
    # --- end FIX 4 ---------------------------------------------------------

    if not TRUE_SUBMISSION:
        # Offline validation is not a scored run. Do not fabricate a submission.parquet here.
        (WORKING_DIR / "OFFLINE_VALIDATION_NOT_SCORED.txt").write_text(
            "This was not a Kaggle competition rerun; no real scored submission was produced.\n"
        )
        print("taaf.kaggle: offline validation only; no fabricated submission.parquet written.")
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

if TRUE_SUBMISSION:
    submission_path = WORKING_DIR / "submission.parquet"
    if not submission_path.is_file():
        raise RuntimeError(
            "Expected the official TAAF teardown to create /kaggle/working/submission.parquet, "
            "but it was not found. No fallback parquet was fabricated. Check setup/run/teardown logs."
        )
    print(f"taaf.kaggle: verified official submission parquet at {submission_path}")


# =============================================================================
# 8. Show the diagnostics
#
# A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is
# rendered inline below (and downloadable from the working directory). You
# should be able to click around through the links.
# =============================================================================

from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html - minimal diagnostics (real submission) suppresses it.")


